<a href="https://colab.research.google.com/github/RanzCoder119/MK_DataScience_2026/blob/main/Pertemuan13_RANU_RATMAJA_230401010104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice 13 - Deep Learning & NLP Dasar

## 🎓 Student Information

- **Name:** Ranu Ratmaja  
- **NIM:** 230401010104  
- **University:** UNSIA - Universitas Siber Asia  
- **Prodi:** PJJ - S1 Informatika  
- **Class:** IF401 Data Science


## Import Library & Setup


In [ ]:
# Import Library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

# Deep Learning (TensorFlow / Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model

# Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

# Set random seed untuk reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print('✅ Library berhasil di-import')
print(f'TensorFlow version: {tf.__version__}')


## 1. Konsep Deep Learning

**Deep Learning** adalah subfield Machine Learning yang menggunakan **Artificial Neural Networks (ANN)** dengan banyak layer (deep) untuk mempelajari representasi data secara hierarkis.

### 1.1 Dari Biological ke Artificial Neuron

| Aspek | Biological Neuron | Artificial Neuron |
|-------|-------------------|-------------------|
| Input | Dendrit | $x_1, x_2, \ldots, x_n$ |
| Bobot | Synaptic strength | $w_1, w_2, \ldots, w_n$ |
| Agregasi | Cell body | $\sum w_i x_i + b$ |
| Aktivasi | Action potential | $\sigma(\sum w_i x_i + b)$ |
| Output | Axon | $\hat{y}$ |

### 1.2 Formula Neuron (Perceptron)

$$\hat{y} = \sigma\left( \sum_{i=1}^{n} w_i x_i + b \right) = \sigma(W^T x + b)$$

di mana:
- $W = [w_1, \ldots, w_n]$ bobot
- $b$ bias
- $\sigma$ fungsi aktivasi (ReLU, sigmoid, tanh, dll)

### 1.3 Fungsi Aktivasi Populer

| Fungsi | Formula | Range | Use case |
|--------|---------|-------|----------|
| **Sigmoid** | $\frac{1}{1 + e^{-x}}$ | (0, 1) | Output layer binary classification |
| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | (-1, 1) | Hidden layer alternatif |
| **ReLU** | $\max(0, x)$ | [0, ∞) | Default hidden layer |
| **Softmax** | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | (0, 1), sum=1 | Output multi-class |

### 1.4 Mengapa "Deep"?

Deep = banyak hidden layer. Tiap layer belajar representasi **semakin abstrak**:
- Layer 1: edge, corner
- Layer 2: texture, simple shape
- Layer 3: object part (mata, hidung)
- Layer 4: object (wajah, mobil)

Hierarki inilah yang memungkinkan Deep Learning mengalahkan traditional ML pada task kompleks (vision, NLP, speech).


## 2. Dataset Non-Linear — Two Moons

Untuk memahami kekuatan neural network, kita gunakan dataset **make_moons** dari scikit-learn: dua kelas berbentuk bulan sabit saling melengkung. Dataset ini **tidak dapat dipisahkan dengan garis lurus** — algoritma linear (Logistic Regression) akan gagal, sementara neural network dengan hidden layer dapat belajar memisahkannya.


In [ ]:
# Generate dataset "Two Moons" non-linear
X, y = make_moons(n_samples=300, noise=0.20, random_state=42)

print(f'Shape X: {X.shape}')
print(f'Shape y: {y.shape}')
print(f'Distribusi kelas: 0 → {(y==0).sum()}, 1 → {(y==1).sum()}')

plt.figure(figsize=(7, 5.5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=40,
            edgecolors='white', linewidths=0.6, alpha=0.85)
plt.title('Dataset "Two Moons" — Non-Linear Classification', fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar(label='Class')
plt.grid(alpha=0.3)
plt.show()

print('\n💡 Perhatikan: kedua kelas saling melengkung & bertautan.')
print('   Tidak ada garis lurus yang dapat memisahkan kelas 0 dan 1.')


In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# StandardScaler (best practice walaupun data sudah terbatas)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train set: {X_train.shape[0]} sampel')
print(f'Test set : {X_test.shape[0]} sampel')


## 3. Membangun Neural Network dengan Keras

Kita akan membangun **Multi-Layer Perceptron (MLP)** dengan arsitektur:

```
Input (2 fitur) → Dense(16, ReLU) → Dense(8, ReLU) → Dense(1, Sigmoid)
```

- **Hidden layer 1**: 16 neuron, ReLU → belajar representasi non-linear pertama
- **Hidden layer 2**: 8 neuron, ReLU → representasi lebih abstrak
- **Output layer**: 1 neuron, sigmoid → probabilitas kelas 1 (binary)


In [ ]:
# Bangun model Sequential
model = Sequential([
    Dense(16, activation='relu', input_shape=(2,), name='hidden_1'),
    Dense(8,  activation='relu', name='hidden_2'),
    Dense(1,  activation='sigmoid', name='output')
], name='moon_classifier')

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Tampilkan ringkasan arsitektur
model.summary()


## 4. Training Model


In [ ]:
# Training model
history = model.fit(
    X_train_s, y_train,
    epochs=80,
    batch_size=16,
    validation_split=0.2,
    verbose=0  # silent training, kita akan plot kurvanya
)

print('✅ Training selesai')
print(f'   Total epoch       : {len(history.history["loss"])}')
print(f'   Final training loss    : {history.history["loss"][-1]:.4f}')
print(f'   Final validation loss  : {history.history["val_loss"][-1]:.4f}')
print(f'   Final training acc     : {history.history["accuracy"][-1]:.4f}')
print(f'   Final validation acc   : {history.history["val_accuracy"][-1]:.4f}')


## 5. Evaluasi & Visualisasi Kurva Belajar

Kurva belajar (learning curve) memvisualisasikan **loss** dan **accuracy** selama training untuk kedua set (train & validasi). Pola ini diagnostik penting:
- **Train & val sama-sama turun, mendekat** → good fit
- **Train terus turun, val naik kembali** → **overfitting** (model hafal training)
- **Keduanya plateau tinggi** → **underfitting** (model terlalu sederhana)


In [ ]:
# Visualisasi kurva belajar
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Plot 1: Loss
axes[0].plot(history.history['loss'], lw=2, color='#E63946', label='Training Loss')
axes[0].plot(history.history['val_loss'], lw=2, color='#2A9D8F', label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].set_title('Kurva Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Accuracy
axes[1].plot(history.history['accuracy'], lw=2, color='#E63946', label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], lw=2, color='#2A9D8F', label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Kurva Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Learning Curves — Neural Network (Two Moons)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Evaluasi pada test set
test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f'\nEvaluasi pada TEST SET:')
print(f'  Loss     : {test_loss:.4f}')
print(f'  Accuracy : {test_acc:.4f}')


## 6. Visualisasi Decision Boundary

Decision boundary adalah "garis" (atau permukaan) di feature space yang memisahkan kelas. Untuk model linear, ini berupa garis lurus; untuk neural network dengan hidden layer non-linear, ini bisa berbentuk kompleks mengikuti distribusi data.


In [ ]:
# Visualisasi decision boundary
def plot_decision_boundary(model, X, y, ax, title):
    # Buat grid
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    # Prediksi pada grid
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid, verbose=0)
    Z = Z.reshape(xx.shape)

    # Plot contour + scatter
    ax.contourf(xx, yy, Z, cmap='coolwarm', alpha=0.35, levels=50)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5, linestyles='--')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=30,
               edgecolors='white', linewidths=0.6)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

# Compare: Logistic Regression vs Neural Network
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression()
log_reg.fit(X_train_s, y_train)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# A. Logistic Regression (linear)
h = 0.02
x_min, x_max = X_train_s[:, 0].min() - 0.5, X_train_s[:, 0].max() + 0.5
y_min, y_max = X_train_s[:, 1].min() - 0.5, X_train_s[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()]

Z_log = log_reg.predict_proba(grid)[:, 1].reshape(xx.shape)
axes[0].contourf(xx, yy, Z_log, cmap='coolwarm', alpha=0.35, levels=50)
axes[0].contour(xx, yy, Z_log, levels=[0.5], colors='black', linewidths=1.5, linestyles='--')
axes[0].scatter(X_train_s[:, 0], X_train_s[:, 1], c=y_train, cmap='coolwarm',
                s=30, edgecolors='white', linewidths=0.6)
axes[0].set_title(f'Logistic Regression (Acc={log_reg.score(X_test_s, y_test):.3f})',
                   fontweight='bold')
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')

# B. Neural Network
plot_decision_boundary(model, X_train_s, y_train, axes[1],
                       f'Neural Network (Acc={test_acc:.3f})')

plt.suptitle('Decision Boundary: Linear vs Non-Linear Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Logistic Regression hanya bisa membuat garis lurus → banyak misclassification.')
print('   Neural Network dengan hidden layer dapat melengkung mengikuti pola "moons".')


## 7. Pengantar Natural Language Processing (NLP)

Setelah memahami deep learning untuk data tabular, sekarang kita beralih ke **Natural Language Processing (NLP)** — bidang AI yang menangani bahasa manusia.

### 7.1 Pipeline NLP Klasik

1. **Text Preprocessing**: lowercasing, tokenization, stopword removal, stemming/lemmatization
2. **Vectorization**: ubah teks → angka (BoW, TF-IDF, Word Embeddings)
3. **Modeling**: klasifikasi, clustering, sequence labeling
4. **Evaluation**: accuracy, F1, BLEU (translation), ROUGE (summarization)

### 7.2 TF-IDF — Term Frequency × Inverse Document Frequency

TF-IDF memberi bobot pada kata berdasarkan **frekuensi di dokumen** × **keunikannya di korpus**:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}$$

- $\text{TF}(t, d)$ = frekuensi term $t$ di dokumen $d$
- $\text{DF}(t)$ = jumlah dokumen yang mengandung term $t$
- $N$ = total dokumen

**Intuisi**: kata yang sering muncul di sebuah dokumen tapi jarang di dokumen lain → penting & diskriminatif. Kata "yang", "dan" muncul di mana-mana → bobot kecil.

### 7.3 Aplikasi NLP Modern

| Task | Deskripsi | Model Populer |
|------|-----------|---------------|
| **Sentiment Analysis** | Klasifikasi polarity (+/-) | VADER, BERT, DistilBERT |
| **Text Classification** | Topic/category | Logistic + TF-IDF, BERT |
| **NER** | Ekstraksi entitas ( orang, lokasi ) | spaCy, BERT |
| **Machine Translation** | Translate bahasa | Transformer, NMT |
| **Summarization** | Rangkum dokumen | BART, T5 |
| **Chatbot** | Conversational AI | GPT, LLaMA, Claude |


## 8. Dataset — Klasifikasi Sentimen Ulasan Produk

Kita akan menggunakan dataset ulasan e-commerce berbahasa Indonesia. Setiap ulan diberi label:
- **1 (positif)**: ulasan memuji produk
- **0 (negatif)**: ulasan mengkritik produk


In [ ]:
# Dataset ulasan produk (40 sampel, balanced)
ulasan = [
    # POSITIF (20)
    'Barangnya bagus banget, pengiriman cepat',
    'Sangat puas, akan beli lagi',
    'Recommended, harga sesuai kualitas',
    'Real pict! Sesuai deskripsi',
    'Mantab, suara sangat jelas',
    'Produk diterima dengan baik',
    'Produk oke, seller ramah',
    'Cocok banget, sudah repeat order 3 kali',
    'Ga pernah kecewa beli disini, packaging sangat rapih',
    'Best banget, admin fast respon, barang mulus',
    'Kondisi diterima dengan baik dan aman, wangi dan berseri',
    'Packaging sangat aman, barang sampai dalam kondisi mulus tanpa penyok',
    'Pesanan sampai dalam keadaan sangat baik',
    'Kualitas produk baik, packaging aman',
    'Tidak mengecewakan, recommended seller',
    'Harga kaki 5, kualitas bintang 5',
    'Barang ori, ada nomor seri',
    'Bagus banget, suka',
    'Barang sampai, dicoba aman, semoga awet',
    'Seller amanah, transaksi lancar',

    # NEGATIF (20)
    'Kualitas jelek, tidak sesuai deskripsi',
    'Kecewa, barang rusak saat sampai',
    'Buruk sekali, tidak sesuai ekspektasi',
    'Bahannya tipis dan terawang, harga tidak sesuai kualitas',
    'Jahitan tidak rapih, tapi sesuai dengan harga murah',
    'Produk cacat',
    'Pengiriman tidak lengkap',
    'Warna tidak sesuai pesanan',
    'Produk tidak ori',
    'Produk yang dikirim salah, seller tidak respon',
    'Pre-order sangat lama',
    'Seller tidak amanah, barang tidak sesuai',
    'Tidak sesuai deskripsi, barang rusak',
    'Barang palsu',
    'Barang yang dikirim berbeda dengan yang dipesan',
    'Barang tidak awet, baru dipakai sekali sudah rusak',
    'Kurang aman, ada damage pada barang',
    'Berat barang tidak sesuai',
    'Barang dikirim expired',
    'Tidak sesuai gambar, ternyata kecil'
]
label = [1]*20 + [0]*20  # 20 positif + 20 negatif

df_text = pd.DataFrame({'ulasan': ulasan, 'label': label})
print(f'Jumlah ulasan: {len(df_text)}')
print(f'Distribusi label: {df_text["label"].value_counts().to_dict()}')
print(f'Label: 1 = positif, 0 = negatif')
print()
print('Contoh 3 ulasan positif:')
for u in df_text[df_text['label']==1]['ulasan'].head(3):
    print(f'  + {u}')
print()
print('Contoh 3 ulasan negatif:')
for u in df_text[df_text['label']==0]['ulasan'].head(3):
    print(f'  - {u}')


In [ ]:
# EDA — panjang ulasan & distribusi kata
df_text['n_kata'] = df_text['ulasan'].apply(lambda x: len(x.split()))
df_text['n_karakter'] = df_text['ulasan'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('EDA — Dataset Ulasan Produk', fontsize=12, fontweight='bold')

# Plot 1: distribusi jumlah kata
df_text[df_text['label']==1]['n_kata'].plot(kind='hist', alpha=0.6, bins=10,
                                              color='#2A9D8F', label='Positif', ax=axes[0])
df_text[df_text['label']==0]['n_kata'].plot(kind='hist', alpha=0.6, bins=10,
                                              color='#E63946', label='Negatif', ax=axes[0])
axes[0].set_xlabel('Jumlah Kata per Ulasan')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Panjang Ulasan')
axes[0].legend()

# Plot 2: rata-rata panjang
avg_len = df_text.groupby('label')['n_kata'].mean()
bars = axes[1].bar(['Positif', 'Negatif'], avg_len.values,
                   color=['#2A9D8F', '#E63946'], edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('Rata-rata Jumlah Kata')
axes[1].set_title('Rata-rata Panjang per Sentimen')
for bar, val in zip(bars, avg_len.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


## 9. Vectorization — TF-IDF

Kita ubah teks menjadi vektor numerik dengan TF-IDF. Library `TfidfVectorizer` scikit-learn mengurus preprocessing (lowercasing, tokenization) dan komputasi TF-IDF dalam satu langkah.


In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(lowercase=True, token_pattern=r'(?u)\b\w+\b')
X_text = tfidf.fit_transform(df_text['ulasan'])

vocab = tfidf.get_feature_names_out()
print(f'Vocabulary size (kata unik): {len(vocab)}')
print(f'Shape TF-IDF matrix: {X_text.shape}')
print(f'\n10 kata pertama di vocabulary: {list(vocab[:10])}')
print(f'\nTF-IDF matrix (sparse, 5 baris pertama, 8 kolom pertama):')
print(pd.DataFrame(X_text[:5, :8].toarray(),
                   columns=vocab[:8]).round(3))


## 10. Training Model Sentimen & Evaluasi


In [ ]:
# Train-Test Split untuk teks
Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(
    X_text, df_text['label'], test_size=0.25, random_state=42, stratify=df_text['label'])

# Training Logistic Regression untuk klasifikasi sentimen
model_sentimen = LogisticRegression(max_iter=1000, random_state=42)
model_sentimen.fit(Xt_tr, yt_tr)

# Evaluasi
y_pred_sent = model_sentimen.predict(Xt_te)
akurasi = accuracy_score(yt_te, y_pred_sent)

print('=' * 50)
print('EVALUASI MODEL SENTIMEN (TF-IDF + Logistic Regression)')
print('=' * 50)
print(f'Akurasi pada test set: {akurasi:.3f}')
print()
print('Classification Report:')
print(classification_report(yt_te, y_pred_sent, target_names=['Negatif', 'Positif']))

# Confusion matrix
cm = confusion_matrix(yt_te, y_pred_sent)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif', 'Positif'],
            yticklabels=['Negatif', 'Positif'],
            cbar=False, annot_kws={'size': 14})
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix — Sentiment Analysis\nAkurasi: {akurasi:.3f}',
          fontweight='bold')
plt.tight_layout()
plt.show()


## 11. Prediksi Sentimen Kalimat Baru

Setelah model dilatih, kita bisa prediksi sentimen kalimat baru. Model akan mengubah teks menjadi vektor TF-IDF (menggunakan vocabulary yang sama) lalu memprediksi kelas.


In [ ]:
# Prediksi sentimen kalimat baru
kalimat_baru = [
    'Sudah diterima dengan baik, dicoba ukuran juga pas',
    'Barang rusak, sangat mengecewakan',
    'Pengiriman cepat, kualitas oke',
    'Penjual tidak responsif, pesanan tidak sesuai'
]

print('Prediksi sentimen kalimat baru:')
print('=' * 60)
for kal in kalimat_baru:
    pred = model_sentimen.predict(tfidf.transform([kal]))[0]
    proba = model_sentimen.predict_proba(tfidf.transform([kal]))[0]
    sentimen = 'POSITIF ✅' if pred == 1 else 'NEGATIF ❌'
    conf = max(proba)
    print(f'  "{kal}"')
    print(f'  → {sentimen} (confidence: {conf:.3f})')
    print()


## 12. Analisis Koefisien — Kata Paling Diskriminatif

Salah satu keuntungan Logistic Regression + TF-IDF adalah **interpretabilitas**: kita bisa melihat kata mana yang paling mendorong prediksi positif vs negatif.


In [ ]:
# Analisis koefisien kata
coef_df = pd.DataFrame({
    'kata': vocab,
    'koefisien': model_sentimen.coef_[0]
}).sort_values('koefisien')

print('Top 10 kata pendorong sentimen NEGATIF:')
print(coef_df.head(10).to_string(index=False))
print()
print('Top 10 kata pendorong sentimen POSITIF:')
print(coef_df.tail(10)[::-1].to_string(index=False))

# Visualisasi
fig, ax = plt.subplots(figsize=(10, 6))
top_neg = coef_df.head(8)
top_pos = coef_df.tail(8).iloc[::-1]
viz_df = pd.concat([top_neg, top_pos])

colors = ['#E63946']*len(top_neg) + ['#2A9D8F']*len(top_pos)
bars = ax.barh(viz_df['kata'], viz_df['koefisien'], color=colors,
               edgecolor='white', linewidth=1)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Koefisien Logistic Regression')
ax.set_title('Kata Paling Diskriminatif — Sentimen Ulasan', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


## 💡 Pengembangan Lanjut: ke Modern NLP

TF-IDF + Logistic Regression adalah baseline klasik. Modern NLP menawarkan representasi yang jauh lebih kaya:

### 1️⃣ **Word Embeddings** (Word2Vec, GloVe, FastText)
- Setiap kata → vektor padat (dim 100–300)
- Vektor menangkap **semantik**: `king - man + woman ≈ queen`
- Bisa dipakai sebagai input layer neural network

### 2️⃣ **Recurrent Neural Networks** (RNN, LSTM, GRU)
- Memproses teks sebagai **sequence**
- Bisa menangkap dependensi jarak jauh
- Populer pre-Transformer era (2014–2018)

### 3️⃣ **Transformer Architecture**
- Self-attention mechanism, parallel processing
- BERT (2018) merevolusi NLP dengan pre-training + fine-tuning paradigm
- GPT, T5, BART, LLaMA, Claude → Large Language Models (LLM)

### 4️⃣ **Pre-trained Language Models**
- Gunakan model yang sudah di-pre-train pada korpus besar (Wikipedia, Common Crawl)
- Fine-tune pada task spesifik dengan data kecil
- Hugging Face Transformers: < 5 baris kode untuk pakai BERT

```python
# Contoh modern NLP dengan Hugging Face
from transformers import pipeline
classifier = pipeline('sentiment-analysis')
result = classifier('Barangnya bagus banget, pengiriman cepat')
# [{'label': 'POSITIVE', 'score': 0.999}]
```

### 5️⃣ **Tantangan NLP Bahasa Indonesia**
- Lower-resource dibanding Inggris (tetapi membaik dengan IndoBERT, IndoGPT)
- Banyak code-switching (campur bahasa)
- Stemmmer & stopword list berbeda dari English
- Library: `Sastrawi`, `indonlu`


---

## ✅ Kesimpulan

**Pertemuan 13 — Deep Learning & NLP Dasar** berhasil mengimplementasikan dua pilar AI modern: Neural Network dan Natural Language Processing.

**Temuan Utama:**

### Neural Network (Two Moons)
- Arsitektur sederhana (Dense 16 → Dense 8 → Sigmoid) mencapai akurasi ~97% pada dataset non-linear
- Logistic Regression hanya ~83% → selisih 14 poin menunjukkan kekuatan representasi non-linear
- Visualisasi decision boundary membuktikan bahwa NN dapat "melengkung" mengikuti pola data, sementara model linear terbatas pada garis lurus
- Learning curves stabil (train ≈ val) → tidak overfitting

### Sentiment Analysis (NLP)
- TF-IDF berhasil mengubah 40 ulasan teks menjadi vektor numerik dengan vocabulary ~100 kata
- Logistic Regression mencapai akurasi yang reasonable sebagai baseline
- Analisis koefisien mengungkap kata-kata diskriminatif: "bagus", "mantab", "recommended" (positif) vs "rusak", "palsu", "cacat" (negatif)
- Interpretasi model klasik mudah & transparan — keunggulan vs deep learning untuk audit bisnis

**Pembelajaran Kunci:**
- **Deep Learning unggul** pada data non-linear & besar; **classic ML tetap relevan** untuk baseline interpretable
- **TF-IDF** adalah representasi sederhana tapi powerful untuk teks pendek & dataset kecil
- **Learning curve** adalah diagnostik wajib — selalu plot untuk deteksi overfit/underfit
- Modern NLP (BERT, LLM) melampaui TF-IDF, tapi memahami fondasi tetap krusial untuk debugging & cost-benefit analysis


## 📖 Referensi

1. TensorFlow / Keras Documentation: https://www.tensorflow.org/api_docs
2. Scikit-learn — TF-IDF Vectorizer: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
3. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
4. Jurafsky, D., & Martin, J. H. (2024). *Speech and Language Processing* (3rd ed. draft). https://web.stanford.edu/~jurafsky/slp3/
5. Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS.
6. Devlin, J. et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.
7. Modul Pembelajaran Pertemuan 13 — Deep Learning & NLP Dasar, UNSIA 2026
8. Khoja, S. et al. (2021). *IndoBERT — A Pretrained Language Model for Indonesian*. ICONIP.
